# 10. 기술 리드: 코드 리뷰, 멘토링, 의사결정 로그

5~10명 규모 팀을 이끌려면 모델 성능만큼 리뷰 기준과 의사결정 기록이 중요합니다.
이 노트북은 멀티모달 ML 코드 리뷰 체크리스트와 ADR(Architecture Decision Record)을 생성합니다.


In [ ]:
from pathlib import Path
import json
import math
import random
import statistics
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print("matplotlib을 불러오지 못했습니다:", exc)

ROOT = Path.cwd()
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

random.seed(42)
np.random.seed(42)

def save_json(name, obj):
    path = ARTIFACTS / name
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def display_df(df, n=20):
    return df.head(n)


In [ ]:
review_items = pd.DataFrame([
    ("data", "회사명/고객명/이메일이 xxx로 마스킹되는가?", "privacy"),
    ("data", "train/valid/test가 session 단위로 분리되는가?", "leakage"),
    ("audio", "샘플레이트, 채널, loudness 정규화가 명시되는가?", "reproducibility"),
    ("ocr", "회전/흐림/저대비 조건별 평가가 있는가?", "robustness"),
    ("eval", "주요 metric 외 failure taxonomy가 있는가?", "research"),
    ("mlops", "run_id, seed, commit, dependency가 기록되는가?", "operation"),
    ("safety", "환경음 경고 조건에서 실행 전 확인을 하는가?", "product"),
], columns=["area", "question", "risk_type"])

review_items.to_csv(ARTIFACTS / "code_review_checklist.csv", index=False, encoding="utf-8-sig")
review_items


In [ ]:
adr = {
    "id": "ADR-001",
    "title": "Use confidence-gated multimodal fusion before end-to-end large model training",
    "status": "proposed",
    "context": "데이터 규모가 작고 안전 요구가 있어 해석 가능한 베이스라인이 필요하다.",
    "decision": "ASR, OCR, 환경음 모듈을 분리 평가하고 confidence gate로 결합한다.",
    "consequences": [
        "오류 원인 분석이 쉽다.",
        "대형 end-to-end 모델보다 성능 상한은 낮을 수 있다.",
        "논문 ablation이 명확해진다.",
    ],
}
save_json("ADR-001.json", adr)
adr


In [ ]:
jp_brief_template = '''
件名: マルチモーダルQA実験の進捗共有

目的:
音声認識、OCR、環境音ゲートを組み合わせ、ノイズ環境でのQA精度と安全性を評価します。

今週の結果:
- ベースライン:
- 改善点:
- 未解決課題:

次のアクション:
- エラー分析を行い、アブレーション実験を追加します。
'''
(ARTIFACTS / "jp_technical_brief_template.md").write_text(dedent(jp_brief_template).strip() + "\n", encoding="utf-8")
print(ARTIFACTS / "jp_technical_brief_template.md")
